run with centroid-env as kernel (problem with ignnition_env)

In [1]:
import sem
import matplotlib.pyplot as plt
import seaborn as sns
import pprint
sns.set_style("whitegrid")
import random
import pandas as pd
import io
import numpy as np


#### Generate input training data 
(will be preprocessed later by migrate.py)

In [36]:
import sem

ns_path = 'ns-3-dev/'
script = 'ai-ns3-mflow-DSCP-numb-ratio-queues-GNN'
campaign_dir = 'testResults' #for training: testResults, for evaluation: testEvaluate
campaign = sem.CampaignManager.new(
    ns_path=ns_path,
    script=script,
    campaign_dir=campaign_dir,
    overwrite=True,
    max_parallel_processes=12
)



Utility functions

In [3]:
def generate_data_rates(start_kbps=100, end_mbps=10, step_kbps=500):
    """
    Generate a list of data rate strings from start to end.

    Args:
        start_kbps (int): starting value in kbps
        end_mbps (int): ending value in Mbps
        step_kbps (int): increment in kbps

    Returns:
        List[str]: list of data rate strings like '1Mbps', '2Mbps'
    """
    rates = []
    for kbps in range(start_kbps, end_mbps * 1000 + 1, step_kbps):
        mbps = kbps / 1000
        if mbps.is_integer():
            rates.append(f"{int(mbps)}Mbps")
        else:
            rates.append(f"{mbps:.3f}Mbps")  # keep decimals if needed
    return rates

# Example usage
data_rates = generate_data_rates()
print(data_rates)


['0.100Mbps', '0.600Mbps', '1.100Mbps', '1.600Mbps', '2.100Mbps', '2.600Mbps', '3.100Mbps', '3.600Mbps', '4.100Mbps', '4.600Mbps', '5.100Mbps', '5.600Mbps', '6.100Mbps', '6.600Mbps', '7.100Mbps', '7.600Mbps', '8.100Mbps', '8.600Mbps', '9.100Mbps', '9.600Mbps']


In [4]:
def generate_delays(start_ms=0.1, end_s=1.0, step_ms=100):
    """
    Generate a list of delay strings from start to end.

    Args:
        start_ms (float): starting value in milliseconds
        end_s (float): ending value in seconds
        step_ms (float): increment in milliseconds

    Returns:
        List[str]: list of delay strings like '0.1ms', '1s'
    """
    delays = []
    current_ms = start_ms
    while current_ms <= end_s * 1000:
        if current_ms < 1:
            delays.append(f"{current_ms:.1f}ms")  # < 1 ms keep decimals
        elif current_ms < 1000:
            delays.append(f"{current_ms/1000:.3f}s")  # between 1 ms and 1 s
        else:
            delays.append(f"{int(current_ms/1000)}s")  # exactly 1 s or more
        current_ms += step_ms
    return delays

# Example usage
delay_list = generate_delays()
print(delay_list)


['0.1ms', '0.100s', '0.200s', '0.300s', '0.400s', '0.500s', '0.600s', '0.700s', '0.800s', '0.900s']


Run to generate training dataset (delete previous training data) 
might take few minutes
If it is stuck near the end just 

In [37]:
#check the input of the datarates and delays
params = {

    'queueDiscType': ['PfifoFast', 'ARED', 'CoDel', 'FqCoDel', 'PIE'],#, 'prio'],
    'queueDiscSize': list(range(1,100,5)),
    'netdevicesQueueSize': list(range(1,100,3)),
    'nVoip': 5,
    'nGaming': 5,
    'nVideo' : 5,
    'nFtp': 5
}
runs = 1 #specify how many randomized experiments we want sem to perform for each parameter combination

campaign.run_missing_simulations(params, runs=runs) 

Running simulations: 100%|██████████| 3300/3300 [01:19<00:00, 41.70simulation/s]


#### Generate input Testing data 
(will be preprocessed later by migrate.py)

In [38]:
ns_path = 'ns-3-dev/'
script = 'ai-ns3-mflow-DSCP-numb-ratio-queues-GNN'
campaign_dir = 'testEvaluate' #for training: testResults, for evaluation: testEvaluate
campaign = sem.CampaignManager.new(
    ns_path=ns_path,
    script=script,
    campaign_dir=campaign_dir,
    overwrite=True,
    max_parallel_processes=12
)

In [39]:
#check the input of the datarates and delays
params = {
    'LinkDataRate':generate_data_rates(),
    'LinkDelay': generate_delays(),
    'queueDiscType': ['PfifoFast', 'ARED', 'CoDel', 'FqCoDel', 'PIE'],
    }


runs = 1 #specify how many randomized experiments we want sem to perform for each parameter combination

campaign.run_missing_simulations(params, runs=runs) 

Running simulations: 100%|██████████| 1000/1000 [00:20<00:00, 48.04simulation/s]


### Some arguments

In [ ]:
'''
    'nVoip': list(range(0, 10, 1)),
    'nGaming': list(range(0, 10, 1)),
    'nVideo' : list(range(0, 10, 1)),
    'nFtp': list(range(0, 10, 1)),

    'ratioVoipEF': np.linspace(0,1,10),
    'ratioVoipAF41': np.linspace(0,1,10),
    'ratioVoipAF42':np.linspace(0,1,10),
    'ratioVoipAF32': np.linspace(0,1,10),
    'ratioVoipAF13': np.linspace(0,1,10),

    'ratioGamingAF41':np.linspace(0,1,10),
    'ratioGamingAF32':np.linspace(0,1,10),
    'ratioGamingAF13':np.linspace(0,1,10),

    'ratioVideoEF': np.linspace(0,1,10),
    'ratioVideoAF33':np.linspace(0,1,10),
    'ratioVideoAF11':np.linspace(0,1,10),
    'ratioVideoBE':np.linspace(0,1,10),

    'ratioFtpAF31':np.linspace(0,1,10),
    'ratioFtpAF11':np.linspace(0,1,10),
    'ratioFtpBE':np.linspace(0,1,10),

    'queueDiscType': ['PfifoFast', 'ARED', 'CoDel', 'FqCoDel', 'PIE', 'prio'],
    'queueDiscSize': list(range(1,100,10)),
    'netdevicesQueueSize': list(range(1,500,10))
    '''